In [92]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Arranging the information 

## 

In [100]:
df = pd.read_csv("data.csv")
X = df

# Change null 
X['Age'].fillna(X['Age'].mean(),inplace=True)
X['RoomService'].fillna(0,inplace=True)
X['FoodCourt'].fillna(0,inplace=True)
X['ShoppingMall'].fillna(0,inplace=True)
X['Spa'].fillna(0,inplace=True)
X['VRDeck'].fillna(0,inplace=True)
X['CryoSleep'].fillna(False,inplace=True)
X['VIP'].fillna(False,inplace=True)
X['Name'].fillna('No Name',inplace=True)

# get GroupId
X['GroupId'] = X['PassengerId'].str.split('_').str[0]
X['LastName'] = X['Name'].str.split(' ').str[1]

X['Cabin'] = X.groupby('GroupId')['Cabin'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Unknown"))
X['Destination'] = X.groupby('GroupId')['Destination'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Unknown"))
X['HomePlanet'] = X.groupby('GroupId')['HomePlanet'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Unknown"))

# get rid from Unknown values
X = X[X['Cabin'] != 'Unknown']
X = X[X['Destination'] != 'Unknown']
X = X[X['HomePlanet'] != 'Unknown']


# Count number of people in each group
group_sizes = X['GroupId'].map(X['GroupId'].value_counts())

# Check if traveling with family (same last name in the same group)
travel_with_family = X.groupby('GroupId')['LastName'].transform(lambda x: x.duplicated(keep=False))


# 0 = alone (group size = 1)
# 1 = group (group size > 1, but no family members)
# 2 = family (group size > 1 and has family members)
X['TravelGroupType'] = 0  # Default: travels alone
X.loc[group_sizes > 1, 'TravelGroupType'] = 1  # Travels with a group
X.loc[travel_with_family, 'TravelGroupType'] = 2  # Travels with family

# Cabin = Deck/Number/Side
X['Deck'] = X['Cabin'].str.split('/').str[0]
X['Number'] = X['Cabin'].str.split('/').str[1]
X['Side'] = X['Cabin'].str.split('/').str[2]

# Transported
Y = X[['Transported']]

# replcace data
X['Destination'] = X['Destination'].replace(['TRAPPIST-1e','PSO J318.5-22','55 Cancri e'],[0,1,2])
X['HomePlanet'] = X['HomePlanet'].replace(['Earth','Europa','Mars'],[0,1,2])
X['VIP'] = X['VIP'].replace([False,True],[0,1]).astype(bool)
X['CryoSleep'] = X['CryoSleep'].replace([False,True],[0,1]).astype(bool)
X['Deck'] = X['Deck'].replace(['A','B','C','D','E','F','G','T'],[1,2,3,4,5,6,7,8])
X['Side'] = X['Side'].replace(['P','S'],[0,1]).astype(bool)
X['Number'] = X['Number'].astype(int)

X = X.drop(['Name','LastName','PassengerId','Transported','Cabin','GroupId'],axis=1)

Y['Transported'] = Y['Transported'].replace([False,True],[0,1])

# Save data in CSV files
X.to_csv("X_data.csv", index=False, sep=";")
Y.to_csv("Y_data.csv", index=False, sep=";")


In [90]:
X

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,TravelGroupType,Deck,Number,Side
0,1,False,0,39.0,False,0.0,0.0,0.0,0.0,0.0,0,2,0,False
1,0,False,0,24.0,False,109.0,9.0,25.0,549.0,44.0,0,6,0,True
2,1,False,0,58.0,True,43.0,3576.0,0.0,6715.0,49.0,2,1,0,True
3,1,False,0,33.0,False,0.0,1283.0,371.0,3329.0,193.0,2,1,0,True
4,0,False,0,16.0,False,303.0,70.0,151.0,565.0,2.0,0,6,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,1,False,2,41.0,True,0.0,6819.0,0.0,1643.0,74.0,0,1,98,False
8689,0,True,1,18.0,False,0.0,0.0,0.0,0.0,0.0,0,7,1499,True
8690,0,False,0,26.0,False,0.0,0.0,1872.0,1.0,0.0,0,7,1500,True
8691,1,False,2,32.0,False,0.0,1049.0,0.0,353.0,3235.0,2,5,608,True
